In [78]:
import xml.etree.ElementTree as ET
import numpy as np
import scipy as sp
from scipy.io import whosmat, loadmat
import statsmodels.api as sm
import pandas as pd

In [3]:
from pathlib import Path

chemin = Path("/mnt/hubel-data-149/Rat012/Rat012_2025-12-15")


In [4]:
for f in sorted(chemin.iterdir()):
    print(f.name)

BeforeCheckSameClu
In
InfraSlowRhythm
IniClu
Rat012_2025-12-15.cat.evt
Rat012_2025-12-15.cat.evt.old
Rat012_2025-12-15.clu.1
Rat012_2025-12-15.clu.10
Rat012_2025-12-15.clu.11
Rat012_2025-12-15.clu.12
Rat012_2025-12-15.clu.12.03.05.2026.14.46
Rat012_2025-12-15.clu.2
Rat012_2025-12-15.clu.3
Rat012_2025-12-15.clu.4
Rat012_2025-12-15.clu.5
Rat012_2025-12-15.clu.6
Rat012_2025-12-15.clu.7
Rat012_2025-12-15.clu.8
Rat012_2025-12-15.clu.9
Rat012_2025-12-15.dat
Rat012_2025-12-15.deltaWaves.events.mat
Rat012_2025-12-15.drowsiness
Rat012_2025-12-15.fet.1
Rat012_2025-12-15.fet.10
Rat012_2025-12-15.fet.11
Rat012_2025-12-15.fet.12
Rat012_2025-12-15.fet.2
Rat012_2025-12-15.fet.3
Rat012_2025-12-15.fet.4
Rat012_2025-12-15.fet.5
Rat012_2025-12-15.fet.6
Rat012_2025-12-15.fet.7
Rat012_2025-12-15.fet.8
Rat012_2025-12-15.fet.9
Rat012_2025-12-15.fil
Rat012_2025-12-15.klg.1
Rat012_2025-12-15.klg.10
Rat012_2025-12-15.klg.11
Rat012_2025-12-15.klg.12
Rat012_2025-12-15.klg.2
Rat012_2025-12-15.klg.3
Rat012_2025-12-

In [5]:
HPC_groups =[7, 8, 9]
MPFC_groups =[10, 11, 12]  

hpc_files =[]
mpfc_files =[]

for g in HPC_groups:
    hpc_files.append({
        "group": g,
        "clu" : next(chemin.glob(f"*.clu.{g}")),
        "res" : next(chemin.glob(f"*.res.{g}"))
    })

for g in MPFC_groups:
    mpfc_files.append({
        "group": g,
        "clu" : next(chemin.glob(f"*.clu.{g}")),
        "res" : next(chemin.glob(f"*.res.{g}"))
    })

print("HPC:")
for x in hpc_files:
    print("HPC groupe", x["group"])
    print("clu:", x["clu"])  
    print("res:", x["res"]) 

print("mPFC:")
for x in mpfc_files:
    print("mPFC groupe", x["group"])
    print("clu:", x["clu"])  
    print("res:", x["res"]) 

HPC:
HPC groupe 7
clu: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.clu.7
res: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.res.7
HPC groupe 8
clu: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.clu.8
res: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.res.8
HPC groupe 9
clu: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.clu.9
res: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.res.9
mPFC:
mPFC groupe 10
clu: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.clu.10
res: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.res.10
mPFC groupe 11
clu: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.clu.11
res: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.res.11
mPFC groupe 12
clu: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.clu.12
res: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.res.12


In [6]:
xml_file = next(chemin.glob("*.xml"))
tree = ET.parse(xml_file)
root = tree.getroot()

sampling_rate = None

for elem in root.iter():
    if elem.tag.lower().endswith("samplingrate"):
        sampling_rate = float(elem.text)
        break

print("fichier xml:", xml_file.name)
print("freq d'ech :", sampling_rate, "Hz")

fichier xml: Rat012_2025-12-15.xml
freq d'ech : 20000.0 Hz


In [7]:
def load_clu_res(clu_path, res_path):

    clu_raw = np.loadtxt(clu_path, dtype=np.int64)
    res = np.loadtxt(res_path, dtype=np.int64)

    clu = clu_raw[1:]

    if len(clu) != len(res):
        raise ValueError(
            f"Longueurs différentes : clu={len(clu)}, res={len(res)}"
        ) 

    return clu, res

In [8]:
FE = 20_000

def load_units(files, fs):
    units ={}

    for x in files:
        group = x["group"]

        clu, res = load_clu_res(x["clu"], x["res"])
        spike_times = res / FE

        for cluster in np.unique(clu):
            units[(group, cluster)]= spike_times[clu == cluster]

    return units

hpc_units = load_units(hpc_files, FE)
mpfc_units = load_units(mpfc_files, FE)     

In [9]:
delta_files =[
    f for f in chemin.iterdir()
    if "delta" in f.name.lower()
] 

for f in delta_files:
    print(f.name)

Rat012_2025-12-15.deltaWaves.events.mat


In [10]:
delta_file= delta_files[0] 

print("Fichier :", delta_file)
print("Taille :", delta_file.stat().st_size / 1024, "KB")

Fichier : /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.deltaWaves.events.mat
Taille : 103.7431640625 KB


In [11]:
with open(delta_file, "rb") as f:
    header = f.read(100)

print(header)

b'MATLAB 5.0 MAT-file, Platform: GLNXA64, Created on: Tue Mar 31 15:44:19 2026                        '


In [12]:
for x in whosmat(delta_file):
    print(x)

('deltaWaves', (1, 1), 'struct')


In [19]:
mat = loadmat(delta_file, simplify_cells=True)

print(mat.keys())

deltawaves = mat["deltaWaves"]

print(type(deltawaves)) 
print(deltawaves.keys())

dict_keys(['__header__', '__version__', '__globals__', 'deltaWaves'])
<class 'dict'>
dict_keys(['timestamps', 'peaks', 'peakNormedPower', 'detectorName', 'troughValue', 'badIntervals'])


In [20]:
for key in ["timestamps", "peaks"]:
    x = np.asarray(deltawaves[key])

    print("\n", key)
    print("shape:", x.shape)
    print("dtype:", x.dtype)
    print("premières valeurs:")
    print(x[:10]) 


 timestamps
shape: (3338, 2)
dtype: float64
premières valeurs:
[[507.7656 508.0152]
 [789.5    789.86  ]
 [802.044  802.316 ]
 [842.6312 842.9016]
 [846.8552 847.1416]
 [861.3624 861.7264]
 [872.0072 872.3392]
 [874.3592 874.6456]
 [876.1632 876.3248]
 [878.7104 879.0536]]

 peaks
shape: (3338,)
dtype: float64
premières valeurs:
[507.8808 789.7328 802.1736 842.7584 846.9912 861.5904 872.1384 874.5032
 876.164  878.8808]


In [21]:
delta_peaks = np.asarray(deltawaves["peaks"], dtype=float)
print("nombre de delta waves:", len(delta_peaks))
print(delta_peaks[:10])

nombre de delta waves: 3338
[507.8808 789.7328 802.1736 842.7584 846.9912 861.5904 872.1384 874.5032
 876.164  878.8808]


In [32]:
def build_hpc_matrix(hpc_units, delta_peaks, window=0.200):
    unit_ids = list(hpc_units.keys())
    X = np.zeros((len(delta_peaks), len(unit_ids)), dtype=int)

    for j, unit_id in enumerate(unit_ids):
        spikes = np.asarray(hpc_units[unit_id])

        left = np.searchsorted(spikes, delta_peaks - window, side = "left")
        right = np.searchsorted(spikes, delta_peaks, side = "right")

        X[:, j] = right - left

    return X, unit_ids 

In [33]:
X_hpc, hpc_unit_ids = build_hpc_matrix (hpc_units, delta_peaks)

print("Shape X_hpc:", X_hpc.shape)
print("Unités HPC:", hpc_unit_ids)

Shape X_hpc: (3338, 61)
Unités HPC: [(7, np.int64(3)), (7, np.int64(7)), (7, np.int64(9)), (7, np.int64(10)), (7, np.int64(13)), (7, np.int64(17)), (7, np.int64(20)), (7, np.int64(25)), (7, np.int64(31)), (7, np.int64(33)), (7, np.int64(35)), (7, np.int64(38)), (7, np.int64(41)), (7, np.int64(42)), (7, np.int64(44)), (7, np.int64(50)), (7, np.int64(51)), (7, np.int64(52)), (7, np.int64(55)), (7, np.int64(58)), (7, np.int64(67)), (7, np.int64(68)), (7, np.int64(71)), (7, np.int64(73)), (7, np.int64(76)), (7, np.int64(80)), (7, np.int64(84)), (7, np.int64(89)), (7, np.int64(96)), (7, np.int64(101)), (7, np.int64(103)), (7, np.int64(104)), (7, np.int64(105)), (7, np.int64(106)), (8, np.int64(5)), (8, np.int64(6)), (8, np.int64(13)), (8, np.int64(23)), (8, np.int64(26)), (8, np.int64(28)), (8, np.int64(29)), (8, np.int64(32)), (8, np.int64(33)), (8, np.int64(45)), (8, np.int64(46)), (9, np.int64(3)), (9, np.int64(7)), (9, np.int64(8)), (9, np.int64(10)), (9, np.int64(11)), (9, np.int64(15)

In [34]:
hpc_units = {
    k: value 
    for k, value in hpc_units.items()
    if k[1] > 1 
} 

mpfc_units = {
    k: value 
    for k, value in mpfc_units.items()
    if k[1] > 1 
} 

In [35]:
print("HPC:", len(hpc_units), "unités")
print("mPFC:", len(mpfc_units), "unités")

print(list(hpc_units.keys()))
print(list(mpfc_units.keys()))

HPC: 61 unités
mPFC: 22 unités
[(7, np.int64(3)), (7, np.int64(7)), (7, np.int64(9)), (7, np.int64(10)), (7, np.int64(13)), (7, np.int64(17)), (7, np.int64(20)), (7, np.int64(25)), (7, np.int64(31)), (7, np.int64(33)), (7, np.int64(35)), (7, np.int64(38)), (7, np.int64(41)), (7, np.int64(42)), (7, np.int64(44)), (7, np.int64(50)), (7, np.int64(51)), (7, np.int64(52)), (7, np.int64(55)), (7, np.int64(58)), (7, np.int64(67)), (7, np.int64(68)), (7, np.int64(71)), (7, np.int64(73)), (7, np.int64(76)), (7, np.int64(80)), (7, np.int64(84)), (7, np.int64(89)), (7, np.int64(96)), (7, np.int64(101)), (7, np.int64(103)), (7, np.int64(104)), (7, np.int64(105)), (7, np.int64(106)), (8, np.int64(5)), (8, np.int64(6)), (8, np.int64(13)), (8, np.int64(23)), (8, np.int64(26)), (8, np.int64(28)), (8, np.int64(29)), (8, np.int64(32)), (8, np.int64(33)), (8, np.int64(45)), (8, np.int64(46)), (9, np.int64(3)), (9, np.int64(7)), (9, np.int64(8)), (9, np.int64(10)), (9, np.int64(11)), (9, np.int64(15)), (9

In [36]:
X_hpc, hpc_unit_ids = build_hpc_matrix (hpc_units, delta_peaks)

print("Dimensions X_hpc:", X_hpc.shape)

Dimensions X_hpc: (3338, 61)


In [39]:
def build_mpfc_targets(mpfc_units, delta_peaks, half_window=0.015):
    unit_ids = list(mpfc_units.keys())

    Y = np.zeros((len(delta_peaks), len(unit_ids)), dtype=int)

    for j, unit_id in enumerate(unit_ids):
        spikes = np.asarray(mpfc_units[unit_id])

        left = np.searchsorted(spikes, delta_peaks - half_window, side = "left")
        right = np.searchsorted(spikes, delta_peaks + half_window, side = "right")

        Y[:, j] = (right > left).astype(int)

    return Y, unit_ids 

In [40]:
Y_mpfc, mpfc_units_ids = build_mpfc_targets (mpfc_units, delta_peaks)

print("X_hpc:", X_hpc.shape)
print("Y_mpfc:", Y_mpfc.shape)

X_hpc: (3338, 61)
Y_mpfc: (3338, 22)


In [41]:
print("Valeurs dans X:", np.unique(X_hpc))
print("Valeurs dans Y:", np.unique(Y_mpfc))

print("NaN dans X:", np.isnan(X_hpc).any())
print("NaN dans Y:", np.isnan(Y_mpfc).any())

Valeurs dans X: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]
Valeurs dans Y: [0 1]
NaN dans X: False
NaN dans Y: False


In [42]:
n_delta_spikes = Y_mpfc.sum(axis=0)

for unit_id, n in zip(mpfc_units_ids, n_delta_spikes):
    print(f"Unité {unit_id}: {n} delta spikes")

Unité (10, np.int64(2)): 6 delta spikes
Unité (10, np.int64(5)): 24 delta spikes
Unité (10, np.int64(24)): 129 delta spikes
Unité (10, np.int64(25)): 29 delta spikes
Unité (11, np.int64(2)): 270 delta spikes
Unité (11, np.int64(4)): 1 delta spikes
Unité (11, np.int64(11)): 16 delta spikes
Unité (11, np.int64(29)): 2 delta spikes
Unité (11, np.int64(33)): 75 delta spikes
Unité (11, np.int64(37)): 14 delta spikes
Unité (11, np.int64(40)): 3 delta spikes
Unité (11, np.int64(41)): 1 delta spikes
Unité (11, np.int64(43)): 18 delta spikes
Unité (11, np.int64(44)): 7 delta spikes
Unité (12, np.int64(8)): 14 delta spikes
Unité (12, np.int64(14)): 1 delta spikes
Unité (12, np.int64(23)): 0 delta spikes
Unité (12, np.int64(26)): 3 delta spikes
Unité (12, np.int64(33)): 2 delta spikes
Unité (12, np.int64(37)): 98 delta spikes
Unité (12, np.int64(38)): 103 delta spikes
Unité (12, np.int64(39)): 14 delta spikes


In [43]:
print("Nombre total de spikes HPC dans les fenêtres:", X_hpc.sum())

spikes_par_hpc_unit = X_hpc.sum(axis=0)

for unit_id, n in zip(hpc_unit_ids, spikes_par_hpc_unit):
    print(f"Unité {unit_id}: {n} spikes dans les fenêtres")

Nombre total de spikes HPC dans les fenêtres: 54520
Unité (7, np.int64(3)): 512 spikes dans les fenêtres
Unité (7, np.int64(7)): 157 spikes dans les fenêtres
Unité (7, np.int64(9)): 2200 spikes dans les fenêtres
Unité (7, np.int64(10)): 125 spikes dans les fenêtres
Unité (7, np.int64(13)): 250 spikes dans les fenêtres
Unité (7, np.int64(17)): 432 spikes dans les fenêtres
Unité (7, np.int64(20)): 365 spikes dans les fenêtres
Unité (7, np.int64(25)): 263 spikes dans les fenêtres
Unité (7, np.int64(31)): 417 spikes dans les fenêtres
Unité (7, np.int64(33)): 3126 spikes dans les fenêtres
Unité (7, np.int64(35)): 172 spikes dans les fenêtres
Unité (7, np.int64(38)): 291 spikes dans les fenêtres
Unité (7, np.int64(41)): 404 spikes dans les fenêtres
Unité (7, np.int64(42)): 714 spikes dans les fenêtres
Unité (7, np.int64(44)): 101 spikes dans les fenêtres
Unité (7, np.int64(50)): 216 spikes dans les fenêtres
Unité (7, np.int64(51)): 1311 spikes dans les fenêtres
Unité (7, np.int64(52)): 436 s

In [48]:
keep_mpfc = Y_mpfc.sum(axis=0) >= 2

Y_mpfc = Y_mpfc[:, keep_mpfc]
mpfc_units_ids = [unit_id for unit_id, keep in zip(mpfc_units_ids, keep_mpfc) if keep] 

keep_hpc = X_hpc.sum(axis=0) > 0

X_hpc = X_hpc[:, keep_hpc]
hpc_unit_ids = [unit_id for unit_id, keep in zip(hpc_unit_ids, keep_hpc) if keep]

print("X_hpc:", X_hpc.shape)
print("Y_mpfc:", Y_mpfc.shape)

X_hpc: (3338, 61)
Y_mpfc: (3338, 18)


In [49]:
n_delta_spikes = Y_mpfc.sum(axis=0)

for unit_id, n in zip(mpfc_units_ids, n_delta_spikes):
    print(unit_id, ":", n)

(10, np.int64(2)) : 6
(10, np.int64(5)) : 24
(10, np.int64(24)) : 129
(10, np.int64(25)) : 29
(11, np.int64(2)) : 270
(11, np.int64(11)) : 16
(11, np.int64(29)) : 2
(11, np.int64(33)) : 75
(11, np.int64(37)) : 14
(11, np.int64(40)) : 3
(11, np.int64(43)) : 18
(11, np.int64(44)) : 7
(12, np.int64(8)) : 14
(12, np.int64(26)) : 3
(12, np.int64(33)) : 2
(12, np.int64(37)) : 98
(12, np.int64(38)) : 103
(12, np.int64(39)) : 14


In [53]:
def make_folds(y, random_state=42):

    rgn = np.random.default_rng(random_state)
    positive_indices = np.where(y==1)[0]
    negative_indices = np.where(y==0)[0]

    n_positive = len(positive_indices)

    if n_positive < 2:
        raise ValueError("Pas assez d'exemples positifs pour faire des folds.")

    rgn.shuffle(positive_indices)
    rgn.shuffle(negative_indices)

    negatives_parts = np.array_split(negative_indices, n_positive)

    folds = []

    for i in range(n_positive):
        test_indices = np.concatenate(([positive_indices[i]], negatives_parts[i]))
        train_indices = np.setdiff1d(np.arange(len(y)), test_indices)
        folds.append((train_indices, test_indices))

    return folds

In [54]:
neuron_id = 0

y = Y_mpfc[:, neuron_id]

print("Neurone:", mpfc_units_ids[neuron_id])
print("Nombre de delta spikes:", y.sum())

folds = make_folds(y)

print("Nombre de folds:", len(folds))

Neurone: (10, np.int64(2))
Nombre de delta spikes: 6
Nombre de folds: 6


In [55]:
for i, (train_indices, test_indices) in enumerate(folds):
    print(f"Fold {i+1}:")
    print("  Train indices:", train_indices)
    print("  Test indices:", test_indices)
    print("  Nombre d'exemples positifs dans le test:", np.sum(y[test_indices]))

Fold 1:
  Train indices: [   0    2    3 ... 3334 3336 3337]
  Test indices: [1910   70  921  125  476 1698 3226  749  855 1719 3325  558    8 3079
 1750  415  352 1522 3335 3093  993  582  623 1906 2178 1704  553 2326
 1174 2461 2167 1013 1942  404 2102 2009 1074 2431 3015 2006  703 2742
 2392   28   17  787  129 1442 2043 1215 3007 2724  514  452 2913  242
 1209 2254  495 1360 3016 1708  175 1713 2480  214 2278 1239 2002 3190
  678 2984 3170 1630  592  850 1309 1438   78 2061  285 2187 1847  454
 1376 3013 1260 2056 2882 1022 2607  773 2531   50 1990 1198 1586 2271
  303 2309  740 1019 1935 3195 1710 1279 2132  192 1068 1695  693  579
 2764 2957 2586 1934 3207  620 1143 2027 2075 2321  223 2639 2506 1753
 3221 1601 2525 2185 1134  178 2844   80 1246 2384 1101 1773 2045 3211
 1097 2477 2474 1950 1315 1569   68 2397 2063 1545 2544 2290 1777 1206
  957 1473 2892  687 2499 1488 2078 1623  365 2035 1034  435 2942 1154
  510  158 1486  400  992  100 2708  466 1441 2950 2864 2920 3147  530


In [57]:
predictions = np.zeros(len(y), dtype=int)

train_indices, test_indices = folds[0]

X_train = X_hpc[train_indices]
X_test = X_hpc[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("Nombre d'exemples positifs dans le train:", np.sum(y_train))
print("Nombre d'exemples positifs dans le test:", np.sum(y_test))

X_train shape: (2781, 61)
X_test shape: (557, 61)
y_train shape: (2781,)
y_test shape: (557,)
Nombre d'exemples positifs dans le train: 5
Nombre d'exemples positifs dans le test: 1


In [61]:
X_train_glm = sm.add_constant(X_train, has_constant='add')
X_test_glm = sm.add_constant(X_test, has_constant='add')

print("X_train avant:", X_train.shape)
print("X_train après:", X_train_glm.shape)
print("X_test avant:", X_test.shape)
print("X_test après:", X_test_glm.shape)

X_train avant: (2781, 61)
X_train après: (2781, 62)
X_test avant: (557, 61)
X_test après: (557, 62)


In [62]:
model = sm.GLM(y_train, X_train_const, family=sm.families.Binomial())
results = model.fit()

print(results.converged)

/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:203: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10

True


In [63]:
pred_test = results.predict(X_test_glm)

print("Nombre de prédictions:", len(pred_test))
print("Minimum:", np.min(pred_test))
print("Maximum:", np.max(pred_test))
print("Nan:", np.isnan(pred_test).any())
print("Inf:", np.isinf(pred_test).any())

print(pred_test[:20])


Nombre de prédictions: 557
Minimum: 0.0
Maximum: 1.0
Nan: False
Inf: False
[2.22536236e-108 3.33201308e-173 9.87869418e-129 6.60922144e-201
 1.08081863e-118 6.96512150e-219 5.47890596e-108 1.43385904e-143
 1.65421321e-044 0.00000000e+000 0.00000000e+000 0.00000000e+000
 3.10871105e-146 2.73284587e-173 1.23807789e-298 1.98673322e-080
 1.16236938e-115 0.00000000e+000 0.00000000e+000 0.00000000e+000]


/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:203: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)


In [64]:
neuron_id = np.argmax(n_delta_spikes)

y = Y_mpfc[:, neuron_id]

print("Neurone:", mpfc_units_ids[neuron_id])
print("Nombre de delta spikes:", y.sum())

Neurone: (11, np.int64(2))
Nombre de delta spikes: 270


In [65]:
folds = make_folds(y)
print("Nombre de folds:", len(folds))

Nombre de folds: 270


In [67]:
train_indices, test_indices = folds[0]

X_train = X_hpc[train_indices]
X_test = X_hpc[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]

X_train_glm = sm.add_constant(X_train, has_constant='add')
X_test_glm = sm.add_constant(X_test, has_constant='add')

model = sm.GLM(y_train, X_train_glm, family=sm.families.Binomial())
results = model.fit()

pred_test = results.predict(X_test_glm)

print("Convergence:", results.converged)
print("Minimum:", np.min(pred_test))
print("Maximum:", np.max(pred_test))
print("Nan:", np.isnan(pred_test).any())
print("Inf:", np.isinf(pred_test).any())
print("Nombre d'exemples positifs dans le train:", np.sum(y_train))
print("Nombre d'exemples positifs dans le test:", np.sum(y_test))

Convergence: True
Minimum: 0.002611922052412819
Maximum: 0.21492947765552875
Nan: False
Inf: False
Nombre d'exemples positifs dans le train: 269
Nombre d'exemples positifs dans le test: 1


In [69]:
predictions = np.zeros(len(y), dtype=float)

for train_indices, test_indices in folds:
    X_train = X_hpc[train_indices]
    X_test = X_hpc[test_indices]

    y_train = y[train_indices]
    y_test = y[test_indices]

    X_train_glm = sm.add_constant(X_train, has_constant='add')
    X_test_glm = sm.add_constant(X_test, has_constant='add')

    model = sm.GLM(y_train, X_train_glm, family=sm.families.Binomial())
    results = model.fit()

    pred_test = results.predict(X_test_glm)

    predictions[test_indices] = results.predict(X_test_glm)

print("Nombre de prédictions:", len(predictions))
print("Minimum:", np.min(predictions))
print("Maximum:", np.max(predictions))
print("Nan:", np.isnan(predictions).any())
print("Inf:", np.isinf(predictions).any())

print("Nombre de valeurs prédites > 0 :", (predictions > 0).sum())

Nombre de prédictions: 3338
Minimum: 0.00015460682586303568
Maximum: 0.5794746873796265
Nan: False
Inf: False
Nombre de valeurs prédites > 0 : 3338


In [70]:
pred_pos = predictions[y == 1]
pred_neg = predictions[y == 0]

print("Nombre de prédictions positives:", len(pred_pos))
print("Nombre de prédictions négatives:", len(pred_neg))

print("Mediane des prédictions si delta spike:", np.median(pred_pos))
print("Mediane des prédictions si pas de delta spike:", np.median(pred_neg))

print("Moyenne des prédictions si delta spike:", np.mean(pred_pos))
print("Moyenne des prédictions si pas de delta spike:", np.mean(pred_neg))

Nombre de prédictions positives: 270
Nombre de prédictions négatives: 3068
Mediane des prédictions si delta spike: 0.08840082236526248
Mediane des prédictions si pas de delta spike: 0.06847045345778868
Moyenne des prédictions si delta spike: 0.10843143392702811
Moyenne des prédictions si pas de delta spike: 0.07878738184828117


In [71]:
errors = np.abs(predictions - y)
e = np.median(errors)

print("Erreur réelle e :", e)
print("Erreur moyenne :", np.mean(errors))

Erreur réelle e : 0.0726551319778615
Erreur moyenne : 0.1445306172409314


In [72]:
rgn = np.random.default_rng(42)

n_shuffles = 1000
shuffled_errors = np.zeros(n_shuffles, dtype=float)

for i in range(n_shuffles):

    shuffled_predictions = rgn.permutation(predictions)
    errors_shuffled = np.abs(shuffled_predictions - y)
    shuffled_errors[i] = np.median(errors_shuffled)

e_shuffled = np.median(shuffled_errors)

g = e_shuffled/e

print("Erreur réelle e :", e)
print("Erreur shufflée e_shuffled :", e_shuffled)
print("Prédiction gain g :", g)

Erreur réelle e : 0.0726551319778615
Erreur shufflée e_shuffled : 0.07359790225356545
Prédiction gain g : 1.0129759626063473


In [76]:
def compute_pred_gain(X, y, n_shuffles=1000, random_state=42):

    folds = make_folds(y, random_state=random_state)
    predictions = np.full(len(y), np.nan, dtype=float)

    n_not_converged = 0

    for train_indices, test_indices in folds:
        X_train = X[train_indices]
        X_test = X[test_indices]

        y_train = y[train_indices]

        X_train_glm = sm.add_constant(X_train, has_constant='add')
        X_test_glm = sm.add_constant(X_test, has_constant='add')

        model = sm.GLM(y_train, X_train_glm, family=sm.families.Binomial())
        results = model.fit()

        if not results.converged:
            n_not_converged += 1
            print("Avertissement: le modèle n'a pas convergé pour un fold.")

    if np.isnan(predictions).any():
        raise ValueError("Certaines prédictions sont NaN")



    errors = np.abs(predictions - y)

    e = np.median(errors)

    rgn = np.random.default_rng(random_state)

    shuffled_errors = np.zeros(n_shuffles, dtype=float)

    for i in range(n_shuffles):
        shuffled_predictions = rgn.permutation(predictions)
    
        shuffled_errors[i] = np.median(errors_shuffled)

    e_shuffled = np.median(shuffled_errors)

    g = e_shuffled / e

    return{
        "gain": g,
        "e": e,
        "e_shuffled": e_shuffled,
        "predictions": predictions,
        "shuffled_errors": shuffled_errors,
        "n_folds": len(folds),
        "n_not_converged": n_not_converged,
        "folds" : folds
    } 

In [79]:
results_hpc = {}  
rows = []

for neuron_id in range (Y_mpfc.shape[1]):
    unit_id = mpfc_units_ids[neuron_id]
    y = Y_mpfc[:, neuron_id]

    print(
        f"Neurone {neuron_id+1}/{Y_mpfc.shape[1]} :"
        f" {unit_id} - {int(y.sum())} delta spikes")

    try:
        result = compute_pred_gain(X_hpc, y, n_shuffles=1000, random_state=42)
        results_hpc[unit_id] = result

        rows.append({
            "neuron_id" : unit_id,
            "n_delta_spikes" : int(y.sum()),
            "n_folds" : result["n_folds"],
            "e": result["e"],
            "e_shuffled": result["e_shuffled"],
            "gain": result["gain"],
            "n_not_converged": result["n_not_converged"]
        })

        print(f"  Gain : {result['gain']:.4f}")

    except Exception as error:
        print(f"Erreur pour le neurone {unit_id}: {error}")

        rows.append({
            "neuron_id" : unit_id,
            "n_delta_spikes" : int(y.sum()),
            "n_folds" : np.nan,
            "e": np.nan,
            "e_shuffled": np.nan,
            "gain": np.nan,
            "n_not_converged": np.nan
        })

df_results_hpc = pd.DataFrame(rows)

Neurone 1/18 : (10, np.int64(2)) - 6 delta spikes


/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/links.py:203: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10

Erreur pour le neurone (10, np.int64(2)): Certaines prédictions sont NaN
Neurone 2/18 : (10, np.int64(5)) - 24 delta spikes
Erreur pour le neurone (10, np.int64(5)): Certaines prédictions sont NaN
Neurone 3/18 : (10, np.int64(24)) - 129 delta spikes
Erreur pour le neurone (10, np.int64(24)): Certaines prédictions sont NaN
Neurone 4/18 : (10, np.int64(25)) - 29 delta spikes
Erreur pour le neurone (10, np.int64(25)): Certaines prédictions sont NaN
Neurone 5/18 : (11, np.int64(2)) - 270 delta spikes
Erreur pour le neurone (11, np.int64(2)): Certaines prédictions sont NaN
Neurone 6/18 : (11, np.int64(11)) - 16 delta spikes
Erreur pour le neurone (11, np.int64(11)): Certaines prédictions sont NaN
Neurone 7/18 : (11, np.int64(29)) - 2 delta spikes


/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return 

Erreur pour le neurone (11, np.int64(29)): Certaines prédictions sont NaN
Neurone 8/18 : (11, np.int64(33)) - 75 delta spikes
Erreur pour le neurone (11, np.int64(33)): Certaines prédictions sont NaN
Neurone 9/18 : (11, np.int64(37)) - 14 delta spikes
Erreur pour le neurone (11, np.int64(37)): Certaines prédictions sont NaN
Neurone 10/18 : (11, np.int64(40)) - 3 delta spikes


...

Erreur pour le neurone (11, np.int64(44)): Certaines prédictions sont NaN
Neurone 13/18 : (12, np.int64(8)) - 14 delta spikes
Erreur pour le neurone (12, np.int64(8)): Certaines prédictions sont NaN
Neurone 14/18 : (12, np.int64(26)) - 3 delta spikes


/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return 

Erreur pour le neurone (12, np.int64(26)): Certaines prédictions sont NaN
Neurone 15/18 : (12, np.int64(33)) - 2 delta spikes


/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
/media/data-103/Laura/DESU/Projet_final/.venv/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return 

Erreur pour le neurone (12, np.int64(33)): Certaines prédictions sont NaN
Neurone 16/18 : (12, np.int64(37)) - 98 delta spikes
Erreur pour le neurone (12, np.int64(37)): Certaines prédictions sont NaN
Neurone 17/18 : (12, np.int64(38)) - 103 delta spikes
Erreur pour le neurone (12, np.int64(38)): Certaines prédictions sont NaN
Neurone 18/18 : (12, np.int64(39)) - 14 delta spikes
Erreur pour le neurone (12, np.int64(39)): Certaines prédictions sont NaN


In [81]:
print(df_results_hpc[["neuron_id", "n_delta_spikes", "gain", "n_not_converged"]])

   neuron_id  n_delta_spikes  gain  n_not_converged
0    (10, 2)               6   NaN              NaN
1    (10, 5)              24   NaN              NaN
2   (10, 24)             129   NaN              NaN
3   (10, 25)              29   NaN              NaN
4    (11, 2)             270   NaN              NaN
5   (11, 11)              16   NaN              NaN
6   (11, 29)               2   NaN              NaN
7   (11, 33)              75   NaN              NaN
8   (11, 37)              14   NaN              NaN
9   (11, 40)               3   NaN              NaN
10  (11, 43)              18   NaN              NaN
11  (11, 44)               7   NaN              NaN
12   (12, 8)              14   NaN              NaN
13  (12, 26)               3   NaN              NaN
14  (12, 33)               2   NaN              NaN
15  (12, 37)              98   NaN              NaN
16  (12, 38)             103   NaN              NaN
17  (12, 39)              14   NaN              NaN
